# Notebook 06 — Visualising Attention

**Goal:** Load the trained model and inspect what its attention heads are doing.

We'll keep the visualisation focused on the most useful views:

1. per-layer, per-head heatmaps
2. average attention by layer
3. attention rollout
4. one simple cell for trying your own text

---

### Contents
1. [Load the model](#1)
2. [Per-head attention heatmaps](#2)
3. [Average attention by layer](#3)
4. [Attention rollout](#4)
5. [Try your own text](#5)
6. [Key takeaways](#6)


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import tiktoken
import sys

sys.path.insert(0, os.path.abspath('.'))
from utils.model import GPTModel
from utils.visualisation import compute_attention_rollout

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('=' * 60)
print('  Notebook 06 — Visualising Attention')
print('=' * 60)


<a id='1'></a>
## 1 — Load the model


In [ ]:
checkpoint_path = 'checkpoints/model.pt'
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError('Run notebook 05 first so checkpoints/model.pt exists.')

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
config = checkpoint['config']
model = GPTModel(config)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

enc = tiktoken.get_encoding('gpt2')

print('Loaded checkpoint:', checkpoint_path)
print('Layers           :', config['n_layers'])
print('Heads            :', config['n_heads'])


In [ ]:
def run_with_attention(text):
    token_ids = enc.encode(text)
    tokens = [enc.decode([t]) for t in token_ids]
    x = torch.tensor([token_ids], dtype=torch.long)
    with torch.no_grad():
        _ = model(x)
    attn = [w.squeeze(0) for w in model.get_attention_weights()]
    return tokens, attn

text = 'A transformer uses self attention to process sequences'
tokens, attn_layers = run_with_attention(text)

print('Tokens:')
print(tokens)


<a id='2'></a>
## 2 — Per-head attention heatmaps

This is the most direct way to inspect what each head is doing.


In [ ]:
for layer_idx, layer_attn in enumerate(attn_layers):
    n_heads = layer_attn.shape[0]
    fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
    if n_heads == 1:
        axes = [axes]

    for head_idx, ax in enumerate(axes):
        sns.heatmap(layer_attn[head_idx].numpy(), annot=len(tokens) <= 8, fmt='.2f', cmap='Blues',
                    xticklabels=tokens, yticklabels=tokens,
                    linewidths=0.4, linecolor='white', square=True, ax=ax)
        ax.set_title(f'Layer {layer_idx}, Head {head_idx}', fontweight='bold', fontsize=10)
        ax.set_xlabel('Key token')
        ax.set_ylabel('Query token')

    plt.tight_layout()
    plt.show()


<a id='3'></a>
## 3 — Average attention by layer

Averaging over heads gives a simpler picture of how a whole layer distributes attention.


In [ ]:
fig, axes = plt.subplots(1, len(attn_layers), figsize=(5 * len(attn_layers), 4))
if len(attn_layers) == 1:
    axes = [axes]

for layer_idx, ax in enumerate(axes):
    avg_attn = attn_layers[layer_idx].mean(dim=0).numpy()
    sns.heatmap(avg_attn, annot=len(tokens) <= 8, fmt='.2f', cmap='Blues',
                xticklabels=tokens, yticklabels=tokens,
                linewidths=0.4, linecolor='white', square=True, ax=ax)
    ax.set_title(f'Layer {layer_idx} (average over heads)', fontweight='bold')
    ax.set_xlabel('Key token')
    ax.set_ylabel('Query token')

plt.tight_layout()
plt.show()


<a id='4'></a>
## 4 — Attention rollout

Raw attention is layer-by-layer. Rollout combines the layers into one view of how information can flow from inputs to outputs.


In [ ]:
rollout = compute_attention_rollout(attn_layers)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(rollout, annot=len(tokens) <= 8, fmt='.2f', cmap='viridis',
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.4, linecolor='white', square=True, ax=ax)
ax.set_title('Attention rollout', fontweight='bold')
ax.set_xlabel('Input token')
ax.set_ylabel('Output position')
plt.tight_layout()
plt.show()


<a id='5'></a>
## 5 — Try your own text

Edit `your_text` and rerun the cell.


In [ ]:
your_text = 'Engineering systems use feedback to control behaviour'

your_tokens, your_attn = run_with_attention(your_text)

print('Tokens:')
print(your_tokens)

avg_attn = your_attn[0].mean(dim=0).numpy()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(avg_attn, annot=len(your_tokens) <= 8, fmt='.2f', cmap='Blues',
            xticklabels=your_tokens, yticklabels=your_tokens,
            linewidths=0.4, linecolor='white', square=True, ax=ax)
ax.set_title('Layer 0 average attention', fontweight='bold')
ax.set_xlabel('Key token')
ax.set_ylabel('Query token')
plt.tight_layout()
plt.show()


<a id='6'></a>
## 6 — Key takeaways

- Per-head heatmaps are the clearest way to inspect learned attention
- Averaging over heads gives a simpler layer-level summary
- Attention rollout gives a rough picture of information flow across layers
- For quick learning, these views are usually enough

If you want, the next refinement could be making the visuals more polished, but I think this is the right level for fast learning.
